<a href="https://colab.research.google.com/github/OmarAyman2005/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OmarAyman2005/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [10]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("HF_TOKEN loaded:", HF_TOKEN is not None)

HF_TOKEN loaded: True


In [11]:
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)

info = api.dataset_info("FlyRank/internship-warehouse")

print("Dataset:", info.id)
print("Access successful!")

Dataset: FlyRank/internship-warehouse
Access successful!


In [12]:
import duckdb

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

MARCH_PATH = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-03/*.parquet"
)

schema_df = con.sql(f"""
DESCRIBE
SELECT *
FROM read_parquet('{MARCH_PATH}')
""").df()

schema_df

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## 1. Unit of analysis + time window

*Unit of analysis: one content-page day per row, identified by report_date, client_hash_id, and content_hash_id.*

*Time window: I will use the March 2026 warehouse partition (month = 2026-03) as a mid-panel development month. This avoids using the final June 2026 sample as a development window.*

*For my refresh-prioritization lane, I will work from fact_content_daily_performance. The output will support ranking content pages for review using observed search and engagement signals.*

In [13]:
grain_check = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS row_count
FROM read_parquet('{MARCH_PATH}')
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 5
""").df()

print("Duplicate grain rows found:", len(grain_check))
grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate grain rows found: 0


,report_date,client_hash_id,content_hash_id,row_count


## 2. Fields: feature / label / context / excluded

*Features: gsc_impressions, gsc_clicks, gsc_avg_position, ga4_sessions, ga4_engaged_sessions*

*Label / proxy: a refresh-priority proxy that will later be derived from observed performance change; it is not used as an input feature.*

*Context: report_date, client_hash_id, and content_hash_id are used for time, grouping, and identification only.*

*Excluded: month is only a partition field, not a model feature. I also exclude any future or label-derived information from the honest feature frame to avoid leakage.*

In [14]:
count_window = con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM read_parquet('{MARCH_PATH}')
""").df()

count_window

,row_count,min_date,max_date
0,9841378,2026-03-01,2026-03-31


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [15]:
availability_check = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
          AND ga4_data_available IS TRUE
    ) AS both_available_rows
FROM read_parquet('{MARCH_PATH}')
""").df()

availability_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available_rows,ga4_available_rows,both_available_rows
0,9841378,3611061,413966,364347


Five-feature frame: I use the first half of March as the information available at the decision moment and the second half only to define a simple future-looking decline proxy. The five features are:

early_impressions — knowable because it uses GSC impressions from March 1–15.
early_clicks — knowable because it uses GSC clicks from March 1–15.
early_avg_position — knowable because it uses observed GSC position from March 1–15.
early_sessions — knowable because it uses GA4 sessions from March 1–15.
early_engaged_sessions — knowable because it uses GA4 engaged sessions from March 1–15.

The proxy label is whether second-half impressions are lower than first-half impressions. This is only a directional refresh-priority proxy, not proof that a page needs refreshing.

In [16]:
feature_df = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,

    SUM(CASE
        WHEN report_date <= DATE '2026-03-15'
        THEN gsc_impressions ELSE 0
    END) AS early_impressions,

    SUM(CASE
        WHEN report_date <= DATE '2026-03-15'
        THEN gsc_clicks ELSE 0
    END) AS early_clicks,

    AVG(CASE
        WHEN report_date <= DATE '2026-03-15'
             AND gsc_avg_position > 0
        THEN gsc_avg_position
    END) AS early_avg_position,

    SUM(CASE
        WHEN report_date <= DATE '2026-03-15'
        THEN ga4_sessions ELSE 0
    END) AS early_sessions,

    SUM(CASE
        WHEN report_date <= DATE '2026-03-15'
        THEN ga4_engaged_sessions ELSE 0
    END) AS early_engaged_sessions,

    SUM(CASE
        WHEN report_date >= DATE '2026-03-16'
        THEN gsc_impressions ELSE 0
    END) AS late_impressions

FROM read_parquet('{MARCH_PATH}')
WHERE gsc_data_available IS TRUE
  AND ga4_data_available IS TRUE
GROUP BY client_hash_id, content_hash_id
""").df()

feature_df["decline_proxy"] = (
    feature_df["late_impressions"] < feature_df["early_impressions"]
).astype(int)

feature_cols = [
    "early_impressions",
    "early_clicks",
    "early_avg_position",
    "early_sessions",
    "early_engaged_sessions"
]

print("Feature-frame rows:", len(feature_df))
print("Five honest features:", feature_cols)
print("\nProxy distribution:")
print(feature_df["decline_proxy"].value_counts())

feature_df[
    ["client_hash_id", "content_hash_id"]
    + feature_cols
    + ["decline_proxy"]
].head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature-frame rows: 63856
Five honest features: ['early_impressions', 'early_clicks', 'early_avg_position', 'early_sessions', 'early_engaged_sessions']

Proxy distribution:
decline_proxy
0    45708
1    18148
Name: count, dtype: int64


,client_hash_id,content_hash_id,early_impressions,early_clicks,early_avg_position,early_sessions,early_engaged_sessions,decline_proxy
0,client_65de48885f4ef01b,content_b1f61fc81b28b2d4,379.0,2.0,4.095154,12.0,0.0,1
1,client_65de48885f4ef01b,content_e25ea7297a1dffd3,2336.0,10.0,4.319753,21.0,1.0,1
2,client_65de48885f4ef01b,content_3c286ded8bd68120,904.0,7.0,8.777106,12.0,1.0,0
3,client_65de48885f4ef01b,content_b2108e8fe3360fa6,330.0,1.0,5.475501,16.0,0.0,1
4,client_65de48885f4ef01b,content_ff867882e604fa96,24.0,0.0,2.850000,2.0,0.0,1


Deliberate leakage test: I now add one feature derived directly from the target on purpose. This feature would not be available legitimately at prediction time, so any large improvement is misleading. I compare the score with and without this leaked feature, then remove it and keep the honest result.

In [17]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score

# Honest features only
X = feature_df[feature_cols].copy()
y = feature_df["decline_proxy"].copy()

# Handle missing values without using future information
imputer = SimpleImputer(strategy="median")
X_clean = imputer.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_clean,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

honest_model = DecisionTreeClassifier(max_depth=3, random_state=42)
honest_model.fit(X_train, y_train)

honest_pred = honest_model.predict(X_test)
honest_accuracy = accuracy_score(y_test, honest_pred)

# Deliberately leaked feature: directly copies the target
leaky_df = feature_df[feature_cols].copy()
leaky_df["label_leak"] = feature_df["decline_proxy"]

X_leaky = SimpleImputer(strategy="median").fit_transform(leaky_df)

Xl_train, Xl_test, yl_train, yl_test = train_test_split(
    X_leaky,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

leaky_model = DecisionTreeClassifier(max_depth=3, random_state=42)
leaky_model.fit(Xl_train, yl_train)

leaky_pred = leaky_model.predict(Xl_test)
leaky_accuracy = accuracy_score(yl_test, leaky_pred)

print(f"Honest accuracy: {honest_accuracy:.3f}")
print(f"Leaky accuracy:  {leaky_accuracy:.3f}")

# Remove the illegal feature
leaky_df = leaky_df.drop(columns=["label_leak"])

print("Leak removed:", "label_leak" not in leaky_df.columns)
print("Final feature count:", leaky_df.shape[1])

Honest accuracy: 0.761
Leaky accuracy:  1.000
Leak removed: True
Final feature count: 5


## 4. Data limits

*This March slice has important limitations. Client history is unbalanced, so not every client has the same amount of usable past data. GA4 availability is much narrower than the full table, so rows with ga4_data_available = FALSE cannot be treated as genuine zero-engagement observations. The proxy label also compares two halves of one month, so it is only a directional development proxy rather than a long-term causal measure of refresh need.*

*These limits mean the results should be used as measured, directional decision support rather than proof that refreshing a page will improve search performance.*

In [18]:
print("March total rows:", 9841378)
print("Rows with both GSC and GA4 available:", 364347)
print("Availability share:", round(364347 / 9841378, 4))

March total rows: 9841378
Rows with both GSC and GA4 available: 364347
Availability share: 0.037


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.